# Week 9 Lab 1 — Micro-step DeepONet and physics-guided zonal loss

<!-- MIE690A article-aligned validation v4 -->

<!-- FLOWMLLAB_COLAB_LAUNCH_V1 -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ehsan-Roohi/FlowMLLab/blob/main/notebooks/week09/W9_Lab1_Microstep_Zonal_DeepONet_Student.ipynb)

**Runtime:** CPU, normally under 2 minutes. **Prerequisites:** case-wise data
splits, relative error, and the basic idea of a neural operator.

This lab turns the Roohi--Mahdavi article *Analysis of the rarefied flow at
micro-step using a DeepONet surrogate model with a physics-guided zonal loss
function* (Microfluidics and Nanofluidics 30:44, published 11 May 2026) into a
controlled classroom experiment.

### Learning outcomes

By the end, you should be able to:

1. distinguish a parameter-to-field operator from pointwise regression;
2. keep every geometry case entirely inside one split;
3. explain why a global mean loss can hide recirculation failure;
4. select a zonal-loss weight using validation cases only; and
5. separate retained article evidence from a manufactured method demonstration.


## Evidence and claim contract — read before code

The paper's DSMC micro-step fields and trained checkpoint are not in the public
repositories audited for this lesson. Therefore:

- the table loaded below is **retained paper evidence**;
- the velocity fields generated in this notebook are **manufactured teaching
  fields**, not DSMC and not a reproduction of the paper;
- the fitted model below is a linearized branch--trunk operator used to expose
  the loss tradeoff, not the paper's neural checkpoint; and
- the held-out ratios 44% and 67% mirror reported article tests, but the
  notebook numbers must never be quoted as article accuracy.

The useful scientific question remains real: when the reverse-flow region is
small, can a globally good operator still be locally unacceptable?


In [ ]:
# FLOWMLLAB_COLAB_BOOTSTRAP_V1
from pathlib import Path as _FlowMLLabPath
import os as _flowmllab_os
import subprocess as _flowmllab_subprocess
import sys as _flowmllab_sys

if "google.colab" in _flowmllab_sys.modules or _flowmllab_os.environ.get("COLAB_RELEASE_TAG"):
    _flowmllab_root = _FlowMLLabPath("/content/FlowMLLab")
    if not (_flowmllab_root / ".git").is_dir():
        _flowmllab_subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/Ehsan-Roohi/FlowMLLab.git", str(_flowmllab_root)],
            check=True,
        )
    _flowmllab_subprocess.run(
        [_flowmllab_sys.executable, "-m", "pip", "install", "-q", "-e", str(_flowmllab_root)],
        check=True,
    )
    _flowmllab_os.chdir(_flowmllab_root / "notebooks/week09")

from pathlib import Path
import json
import platform
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ModuleNotFoundError:
    display = print

REPO_ROOT = next(
    candidate for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "results/mahdavi_deeponet").is_dir()
)
RESULTS = REPO_ROOT / "results/mahdavi_deeponet"
plt.rcParams.update({"font.size": 11, "axes.labelsize": 12, "legend.fontsize": 9})
print("Python:", platform.python_version())
print("FlowMLLab root:", REPO_ROOT)


In [ ]:
from flowmllab.mahdavi_deeponet import (
    manufactured_step_velocity,
    zonal_velocity_metrics,
)

paper = pd.read_csv(RESULTS / "step_paper_evidence.csv")
display(paper.pivot(index="objective", columns="scope", values="reported_error_percent"))


## 1. DeepONet as a parameter-to-field map

For a geometry parameter $h/H$ and query coordinate $\mathbf{y}=(x,y)$, a
DeepONet has the separable form

$$
\widehat{G}(h/H)(\mathbf{y})=
\sum_{k=1}^{r} b_k(h/H)\,t_k(\mathbf{y})+b_0.
$$

The **branch** network encodes the input case; the **trunk** network encodes the
query coordinate. All points from one height are correlated parts of a single
operator sample. Randomly splitting points would put the same geometry in both
training and test sets and produce leakage.

The paper defines the recirculation zone from the reference streamwise
velocity, $U<0$, and balances two separately normalized regional errors:

$$
\mathcal L_{\rm zonal}=\alpha\mathcal L_{U<0}+
(1-\alpha)\mathcal L_{U\ge 0}.
$$

Predict before running: as $\alpha$ increases, which error should decrease,
and what global tradeoff might appear?


In [ ]:
x = np.linspace(0.0, 5.0, 72)
y = np.linspace(0.0, 1.0, 32)
xx, yy = np.meshgrid(x, y)
u_demo, v_demo, solid_demo = manufactured_step_velocity(xx, yy, 0.44)

masked_u = np.ma.array(u_demo, mask=solid_demo)
fig, ax = plt.subplots(figsize=(10.5, 3.0), constrained_layout=True)
levels = np.linspace(-0.7, 1.55, 24)
contour = ax.contourf(xx, yy, masked_u, levels=levels, cmap="coolwarm", extend="both")
ax.contour(xx, yy, masked_u, levels=[0.0], colors="black", linewidths=1.1)
ax.fill_between([0, 1], 0, 0.44, color="0.2", label="solid step")
ax.set(xlabel=r"$x/H$", ylabel=r"$y/H$", title="Manufactured teaching field, $h/H=0.44$")
ax.set_aspect("equal", adjustable="box")
fig.colorbar(contour, ax=ax, label=r"$U/U_0$")
plt.show()

print("This plot is pedagogical, not article DSMC data.")


## 2. A reviewable linear branch--trunk operator

To isolate the effect of the objective, we use polynomial branch features of
$h/H$ and a fixed, low-rank spatial trunk. Their outer products form a
separable operator basis. Weighted least squares then changes only the loss,
not the data, basis, optimizer tolerance, or split.

This is the linear limit of the branch--trunk idea. An assignment extension at
the end replaces both feature maps with small neural networks while preserving
the same split and metrics.


In [ ]:
def operator_design(height_ratio):
    xn = xx / x.max()
    local = [
        np.exp(-((xx - xc) / sx) ** 2 - (yy / sy) ** 2)
        for xc, sx, sy in ((1.25, .45, .16), (1.70, .65, .22), (2.20, .90, .28))
    ]
    trunk = np.stack([
        np.ones_like(xx), xn, yy, xn**2, yy**2, xn * yy,
        np.sin(np.pi * yy), np.cos(np.pi * yy), *local,
        xx * local[1], yy * local[1],
    ], axis=-1)
    branch = np.array([1.0, height_ratio, height_ratio**2, height_ratio**3])
    return (trunk[..., None, :] * branch[None, None, :, None]).reshape(*xx.shape, -1)


def fit_operator(case_heights, alpha=None):
    designs, u_targets, v_targets, vortex_flags = [], [], [], []
    for height in case_heights:
        u, v, solid = manufactured_step_velocity(xx, yy, height)
        valid = ~solid
        designs.append(operator_design(height)[valid])
        u_targets.append(u[valid])
        v_targets.append(v[valid])
        vortex_flags.append(u[valid] < 0.0)
    design = np.vstack(designs)
    u_target = np.concatenate(u_targets)
    v_target = np.concatenate(v_targets)
    vortex = np.concatenate(vortex_flags)
    if alpha is None:
        weight = np.full(len(u_target), 1.0 / len(u_target))
    else:
        weight = np.where(
            vortex, alpha / vortex.sum(), (1.0 - alpha) / (~vortex).sum()
        )
    normal = design.T @ (weight[:, None] * design) + 1e-7 * np.eye(design.shape[1])
    coef_u = np.linalg.solve(normal, design.T @ (weight * u_target))
    coef_v = np.linalg.solve(normal, design.T @ (weight * v_target))
    return coef_u, coef_v


def evaluate_operator(coefficients, case_heights):
    coef_u, coef_v = coefficients
    rows = []
    for height in case_heights:
        u, v, solid = manufactured_step_velocity(xx, yy, height)
        design = operator_design(height)
        metrics = zonal_velocity_metrics(
            u, v, design @ coef_u, design @ coef_v,
            alpha=0.7, valid_mask=~solid,
        )
        rows.append({"h_over_H": height, **metrics})
    return pd.DataFrame(rows)


## 3. Freeze the split and selection rule

We reserve 44% and 67% for the final teaching test. Heights 40% and 60% are
validation cases. The remaining six cases fit the operator. No spatial point
from a reserved geometry enters fitting.

**Predeclared rule:** among $\alpha\in\{0.3,0.5,0.6,0.7,0.8\}$, choose the
largest vortex improvement whose validation global relative error is no more
than two percentage points worse than the unweighted fit. This prevents a
zonal win purchased by an unlimited global failure.


In [ ]:
all_heights = np.array([.25, .30, .35, .40, .44, .50, .55, .60, .67, .72])
test_heights = np.array([.44, .67])
validation_heights = np.array([.40, .60])
development_heights = np.array([
    value for value in all_heights
    if value not in set(test_heights) | set(validation_heights)
])
print("development:", development_heights)
print("validation:", validation_heights)
print("unopened teaching test:", test_heights)

baseline_validation = evaluate_operator(
    fit_operator(development_heights, alpha=None), validation_heights
)
baseline_global = 100 * baseline_validation["full_relative_l2"].mean()
selection_rows = []
for alpha in (.3, .5, .6, .7, .8):
    metrics = evaluate_operator(fit_operator(development_heights, alpha), validation_heights)
    selection_rows.append({
        "alpha": alpha,
        "validation_global_percent": 100 * metrics["full_relative_l2"].mean(),
        "validation_vortex_percent": 100 * metrics["vortex_relative_l2"].mean(),
    })
selection = pd.DataFrame(selection_rows)
eligible = selection[
    selection["validation_global_percent"] <= baseline_global + 2.0
]
selected_alpha = float(eligible.sort_values("validation_vortex_percent").iloc[0]["alpha"])
display(selection)
print(f"unweighted validation global error: {baseline_global:.3f}%")
print("selected alpha:", selected_alpha)
assert selected_alpha == 0.6


## Stop: teaching-test gate

At this point the split, basis, regularization, candidate weights, and selection
rule are frozen. Write your prediction for the two reserved heights. Only then
run the next cell. If you change a choice after viewing the result, these cases
become development data and must no longer be called held out.


In [ ]:
fit_heights = np.concatenate([development_heights, validation_heights])
global_test = evaluate_operator(fit_operator(fit_heights, None), test_heights)
zonal_test = evaluate_operator(fit_operator(fit_heights, selected_alpha), test_heights)

comparison = pd.DataFrame({
    "model": ["unweighted", "zonal"],
    "mean_global_percent": [
        100 * global_test["full_relative_l2"].mean(),
        100 * zonal_test["full_relative_l2"].mean(),
    ],
    "mean_vortex_percent": [
        100 * global_test["vortex_relative_l2"].mean(),
        100 * zonal_test["vortex_relative_l2"].mean(),
    ],
})
display(comparison)

fig, ax = plt.subplots(figsize=(7.4, 4.1), constrained_layout=True)
locations = np.arange(2)
width = 0.34
ax.bar(locations - width / 2, comparison["mean_global_percent"], width, label="global")
ax.bar(locations + width / 2, comparison["mean_vortex_percent"], width, label="recirculation")
ax.set_xticks(locations, comparison["model"])
ax.set(ylabel="mean relative L2 error (%)", title="Manufactured held-out cases: objective tradeoff")
ax.legend(frameon=False)
ax.grid(axis="y", alpha=.25)
plt.show()


## 4. Interpret without mixing evidence levels

The manufactured experiment should show the intended mechanism: the selected
zonal objective reduces reverse-flow error while accepting a modest increase
in whole-field error. The paper reports the same qualitative tradeoff on its
research data: zonal loss changes the reported recirculation-zone error from
14.6135% (MSE) to 11.9413%, while the full-domain value changes from 2.1739%
to 2.2254%.

Those percentages came from the article table, not the bars above. The
notebook's role is to let you inspect *why* the tradeoff occurs and *how* to
select a weight without opening the final cases.

### DeepONet implementation exercise

Replace `operator_design` with two small Keras networks:

- branch input: one scalar, $h/H$;
- trunk input: two scalars, $(x/H,y/H)$;
- output: dot product of equal-width branch and trunk vectors, with separate
  heads for $U$ and $V$.

Keep complete geometry cases together. Implement the regional means before
mixing them with $\alpha$; a pointwise weight without regional normalization is
not the same objective. Compare a standard DeepONet with a larger fusion model
under the same sparse case budget—the paper reports that additional capacity
can overfit when only seven cases are available.

### Required submission

1. a signed split table;
2. the validation-only $\alpha$ sweep;
3. global and reverse-flow metrics for every held-out geometry;
4. one contour locating the largest local error; and
5. one paragraph stating which outputs are manufactured and which are retained
   article evidence.
